In [ ]:
import time
import numpy as np
import torch
import h5py

from models.cmlp import cMLP
from models.cmlp import train_model_ista

# ---------------------------
# 0. 디바이스 설정
# ---------------------------
device = torch.device('cuda:6')
print(f"Using device: {device}")

# ---------------------------
# 1. 데이터 로드 (h5)
# ---------------------------
h5_path = "simulated/260420_data/four_nodes/signal.h5"

with h5py.File(h5_path, 'r') as f:
    X_np = f['signal'][()].astype(np.float32)  # (N_trials, N_features, T) = (2000, 4, 2000)

X_np = X_np.transpose(0, 2, 1)   # (N_trials, T, N_features) = (2000, 2000, 4)
print(f"X_np shape: {X_np.shape}")

X = torch.tensor(X_np).to(device)
print(f"X tensor shape: {X.shape}, device: {X.device}")

# ---------------------------
# 2. 하이퍼파라미터
# ---------------------------
p   = X.shape[-1]   # 3
lag = 1
hidden = [100]
lam = 0.1
lr  = 1e-2
max_iter    = 2000
check_every = 100

# ---------------------------
# 3. 모델 생성
# ---------------------------
cmlp = cMLP(
    num_series=p,
    lag=lag,
    hidden=hidden,
    activation='relu'
).to(device)

# ---------------------------
# 4. 학습 + 시간 측정
# ---------------------------
start_time = time.time()

loss_history = train_model_ista(
    cmlp,
    X,
    lam=lam,
    lr=lr,
    max_iter=max_iter,
    check_every=check_every,
    verbose=1
)

end_time     = time.time()
elapsed_time = end_time - start_time

print(f"\nTraining time: {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} min)")

GC = cmlp.GC().cpu().data.numpy()
print("Estimated Granger Causality Matrix:")
print(GC)

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------
# 저장 폴더
# ---------------------------
save_dir = "results_four_nodes_260420"
os.makedirs(save_dir, exist_ok=True)

# 모델 저장
torch.save(cmlp.state_dict(), os.path.join(save_dir, "cmlp_model.pt"))

# 학습 시간 저장
np.save(os.path.join(save_dir, "elapsed_time.npy"), np.array(elapsed_time))

# loss_history 저장
if loss_history is not None:
    try:
        loss_history_np = np.array([float(x) for x in loss_history])
        np.save(os.path.join(save_dir, "loss_history.npy"), loss_history_np)
    except:
        pass

# Binary GC matrix
GC = cmlp.GC().cpu().data.numpy()
np.save(os.path.join(save_dir, "gc_binary.npy"), GC)

# Continuous GC strength matrix
def extract_gc_strength(cmlp):
    strengths = []
    for net in cmlp.networks:
        first_conv = None
        for module in net.modules():
            if isinstance(module, torch.nn.Conv1d):
                first_conv = module
                break
        if first_conv is None:
            raise RuntimeError("Conv1d layer를 찾지 못했습니다.")
        w = first_conv.weight.detach().cpu()
        s = torch.norm(w, dim=(0, 2))
        strengths.append(s.numpy())
    return np.stack(strengths, axis=0)

GC_strength = extract_gc_strength(cmlp)
np.save(os.path.join(save_dir, "gc_strength.npy"), GC_strength)

print("Binary GC matrix:")
print(GC)
print("\nContinuous GC strength matrix:")
print(np.round(GC_strength, 4))
print(f"\nSaved to: {save_dir}")
print(f"Training time: {elapsed_time:.2f} sec")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

save_dir = "results_four_nodes_260420"

GC           = np.load(f"{save_dir}/gc_binary.npy")
GC_strength  = np.load(f"{save_dir}/gc_strength.npy")
elapsed_time = np.load(f"{save_dir}/elapsed_time.npy")

print("Binary GC matrix:")
print(GC)
print("\nContinuous GC strength matrix:")
print(np.round(GC_strength, 4))
print(f"\nTraining time: {float(elapsed_time):.2f} sec ({float(elapsed_time)/60:.2f} min)")

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
sns.heatmap(
    GC,
    annot=True,
    cmap="Blues",
    fmt=".0f",
    linewidths=.5,
    xticklabels=[1, 2, 3, 4],
    yticklabels=[1, 2, 3, 4]
)
plt.title(f"Binary Granger Causality Matrix\nTime: {float(elapsed_time):.2f} sec")

plt.subplot(1, 2, 2)
sns.heatmap(
    GC_strength,
    annot=True,
    cmap="YlGnBu",
    fmt=".2f",
    linewidths=.5,
    xticklabels=[1, 2, 3, 4],
    yticklabels=[1, 2, 3, 4]
)
plt.title(f"Continuous GC Strength\nTime: {float(elapsed_time):.2f} sec")

plt.tight_layout()
plt.show()

# loss plot
loss_path = f"{save_dir}/loss_history.npy"
if os.path.exists(loss_path):
    loss_history = np.load(loss_path)
    check_every  = 100
    iterations   = np.arange(1, len(loss_history) + 1) * check_every

    plt.figure(figsize=(7, 4))
    plt.plot(iterations, loss_history)
    plt.title(f"Training Loss\nTime: {float(elapsed_time):.2f} sec")
    plt.xlabel("Iteration")
    plt.ylabel("Loss")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("loss_history.npy 없음")